<a href="https://colab.research.google.com/github/smartinternz02/SI-GuidedProject-580322-1694799104/blob/main/indusdata.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
!pip install PyMuPDF opencv-python pillow tqdm matplotlib


In [9]:
# CELL 2: Import Libraries
import os
import numpy as np
import cv2
import matplotlib.pyplot as plt
from PIL import Image
import fitz  # PyMuPDF
from google.colab import files
import re
from tqdm.notebook import tqdm
import io
import shutil

In [8]:
def upload_pdf():
    """Upload PDF file to Colab"""
    print("Please upload your PDF file...")
    uploaded = files.upload()
    pdf_path = list(uploaded.keys())[0]
    print(f"PDF '{pdf_path}' uploaded successfully!")
    return pdf_path

def extract_images_from_pdf(pdf_path, start_page, end_page):
    """Extract all images from the PDF within the specified page range"""
    pdf_document = fitz.open(pdf_path)
    total_pages = pdf_document.page_count

    # Validate page range
    start_page = max(1, min(start_page, total_pages))
    end_page = min(end_page, total_pages)

    print(f"Processing pages {start_page} to {end_page} out of {total_pages} total pages...")

    # Create output directory
    output_dir = "extracted_images"
    os.makedirs(output_dir, exist_ok=True)

    # Process each page
    extracted_images = []

    for page_num in tqdm(range(start_page-1, end_page)):
        page = pdf_document.load_page(page_num)

        # Get page as an image
        pix = page.get_pixmap(matrix=fitz.Matrix(300/72, 300/72))  # 300 DPI rendering
        img_data = pix.samples
        img = Image.frombytes("RGB", [pix.width, pix.height], img_data)
        img_np = np.array(img)

        # Save the full page temporarily
        page_path = f"{output_dir}/page_{page_num+1}.png"
        img.save(page_path)
        extracted_images.append({
            'page_num': page_num+1,
            'path': page_path,
            'image': img_np
        })

    pdf_document.close()
    return extracted_images

def display_image(img, figsize=(10, 10)):
    """Display an image using matplotlib"""
    plt.figure(figsize=figsize)
    plt.imshow(img)
    plt.axis('off')
    plt.show()


In [7]:
# CELL 4: Define Detection and Processing Functions
def detect_seals(img, min_area=1000, max_area=500000):
    """Detect potential seal regions in the image"""
    # Convert to grayscale
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)

    # Apply adaptive thresholding to handle variations in background
    binary = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                  cv2.THRESH_BINARY_INV, 11, 2)

    # Find contours
    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    # Filter contours by area
    valid_contours = []
    for cnt in contours:
        area = cv2.contourArea(cnt)
        if min_area < area < max_area:
            x, y, w, h = cv2.boundingRect(cnt)
            # Filter out extreme aspect ratios
            aspect_ratio = w / h if h > 0 else 0
            if 0.2 < aspect_ratio < 5:  # Filter out very narrow or wide rectangles
                valid_contours.append(cnt)

    return valid_contours

def extract_label(img, box, padding=10):
    """Extract potential label near the seal image"""
    x, y, w, h = box

    # Look for label below the seal (most common position)
    label_region = img[y+h:y+h+50, max(0, x-20):min(img.shape[1], x+w+20)]

    return label_region


In [6]:
# CELL 5: Define Interactive Processing Function
def process_pages(extracted_images, output_dir="extracted_seals"):
    """Process each page to identify and extract seal images with their labels"""
    os.makedirs(output_dir, exist_ok=True)

    seal_count = 0
    processed_images = []

    # Process each page
    for page_data in extracted_images:
        img = page_data['image']
        page_num = page_data['page_num']

        # Display the page
        print(f"\nProcessing Page {page_num}")
        display_image(img, figsize=(15, 20))

        # Automatically detect potential seal images
        contours = detect_seals(img)

        # Draw boxes around detected regions
        img_with_boxes = img.copy()
        for i, cnt in enumerate(contours):
            x, y, w, h = cv2.boundingRect(cnt)
            cv2.rectangle(img_with_boxes, (x, y), (x+w, y+h), (0, 255, 0), 3)
            cv2.putText(img_with_boxes, f"{i+1}", (x, y-10),
                       cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

        print("Detected potential seal regions:")
        display_image(img_with_boxes, figsize=(15, 20))

        # Manual selection process
        print("Do you want to extract these regions? (yes/no/custom)")
        selection = input().strip().lower()

        if selection == "yes":
            # Extract all detected regions
            for i, cnt in enumerate(contours):
                x, y, w, h = cv2.boundingRect(cnt)

                # Add padding around the seal
                padding = 20
                x1 = max(0, x - padding)
                y1 = max(0, y - padding)
                x2 = min(img.shape[1], x + w + padding)
                y2 = min(img.shape[0], y + h + padding)

                seal_img = img[y1:y2, x1:x2]

                # Try to extract label
                label_img = extract_label(img, (x, y, w, h))

                # Save the seal image
                seal_path = f"{output_dir}/seal_{page_num}_{i+1}.png"
                cv2.imwrite(seal_path, cv2.cvtColor(seal_img, cv2.COLOR_RGB2BGR))

                # Save the label image
                label_path = f"{output_dir}/seal_{page_num}_{i+1}_label.png"
                cv2.imwrite(label_path, cv2.cvtColor(label_img, cv2.COLOR_RGB2BGR))

                processed_images.append({
                    'page_num': page_num,
                    'seal_num': i+1,
                    'seal_path': seal_path,
                    'label_path': label_path
                })

                seal_count += 1

        elif selection == "custom":
            print("Enter the regions to extract (comma-separated numbers, e.g., '1,3,4'):")
            selected = input().strip()
            selected_indices = [int(idx) - 1 for idx in selected.split(',') if idx.strip().isdigit()]

            for idx in selected_indices:
                if 0 <= idx < len(contours):
                    cnt = contours[idx]
                    x, y, w, h = cv2.boundingRect(cnt)

                    # Add padding around the seal
                    padding = 20
                    x1 = max(0, x - padding)
                    y1 = max(0, y - padding)
                    x2 = min(img.shape[1], x + w + padding)
                    y2 = min(img.shape[0], y + h + padding)

                    seal_img = img[y1:y2, x1:x2]

                    # Try to extract label
                    label_img = extract_label(img, (x, y, w, h))

                    # Save the seal image
                    seal_path = f"{output_dir}/seal_{page_num}_{idx+1}.png"
                    cv2.imwrite(seal_path, cv2.cvtColor(seal_img, cv2.COLOR_RGB2BGR))

                    # Save the label image
                    label_path = f"{output_dir}/seal_{page_num}_{idx+1}_label.png"
                    cv2.imwrite(label_path, cv2.cvtColor(label_img, cv2.COLOR_RGB2BGR))

                    processed_images.append({
                        'page_num': page_num,
                        'seal_num': idx+1,
                        'seal_path': seal_path,
                        'label_path': label_path
                    })

                    seal_count += 1

        # Option to manually select regions by drawing boxes
        print("Do you want to manually select regions? (yes/no)")
        manual_select = input().strip().lower()

        if manual_select == "yes":
            print("Enter coordinates for manual selection (x,y,width,height):")
            print("Enter 'done' when finished")

            manual_count = 0
            while True:
                coords = input().strip()
                if coords.lower() == 'done':
                    break

                try:
                    x, y, w, h = map(int, coords.split(','))

                    # Add padding around the seal
                    padding = 20
                    x1 = max(0, x - padding)
                    y1 = max(0, y - padding)
                    x2 = min(img.shape[1], x + w + padding)
                    y2 = min(img.shape[0], y + h + padding)

                    seal_img = img[y1:y2, x1:x2]

                    # Save the seal image
                    manual_count += 1
                    seal_path = f"{output_dir}/seal_{page_num}_manual_{manual_count}.png"
                    cv2.imwrite(seal_path, cv2.cvtColor(seal_img, cv2.COLOR_RGB2BGR))

                    processed_images.append({
                        'page_num': page_num,
                        'seal_num': f"manual_{manual_count}",
                        'seal_path': seal_path
                    })

                    seal_count += 1
                except:
                    print("Invalid coordinates. Please use format 'x,y,width,height'")

    print(f"\nExtraction complete. Total {seal_count} seal images extracted.")
    return processed_images

In [5]:
# CELL 6: Define Enhanced Extraction Functions
def enhanced_seal_extraction(extracted_images, output_dir="extracted_seals"):
    """More sophisticated extraction of seals and their labels"""
    os.makedirs(output_dir, exist_ok=True)
    extracted_count = 0

    for page_data in extracted_images:
        img = page_data['image']
        page_num = page_data['page_num']

        # Convert to grayscale
        gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)

        # Apply thresholding to separate foreground from background
        _, binary = cv2.threshold(gray, 240, 255, cv2.THRESH_BINARY_INV)

        # Morphological operations to clean up the binary image
        kernel = np.ones((5,5), np.uint8)
        binary = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel)
        binary = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel)

        # Find contours
        contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        # Filter and process contours
        for i, cnt in enumerate(contours):
            area = cv2.contourArea(cnt)
            if 3000 < area < 300000:  # Filter by size
                x, y, w, h = cv2.boundingRect(cnt)
                aspect_ratio = w / h if h > 0 else 0

                # Filter by aspect ratio
                if 0.5 < aspect_ratio < 2.5:
                    # Extract the region with some padding
                    padding = 20
                    x1 = max(0, x - padding)
                    y1 = max(0, y - padding)
                    x2 = min(img.shape[1], x + w + padding)
                    y2 = min(img.shape[0], y + h + padding)

                    # Extract the region
                    seal_img = img[y1:y2, x1:x2]

                    # Save the image
                    img_filename = f"{output_dir}/seal_{page_num}_{i+1}.png"
                    cv2.imwrite(img_filename, cv2.cvtColor(seal_img, cv2.COLOR_RGB2BGR))
                    extracted_count += 1

    print(f"Automatic extraction complete. {extracted_count} potential seals extracted.")
    return output_dir

def manual_cropping_interface(extracted_images, output_dir="manually_extracted_seals"):
    """Interactive manual cropping interface"""
    os.makedirs(output_dir, exist_ok=True)
    extracted_count = 0

    for page_data in tqdm(extracted_images):
        img = page_data['image']
        page_num = page_data['page_num']

        # Display the page
        print(f"\nPage {page_num}:")
        display_image(img, figsize=(15, 20))

        print("Enter coordinates to crop (x,y,width,height) or 'skip' to move to next page:")
        while True:
            coords = input().strip()
            if coords.lower() == 'skip':
                break

            try:
                x, y, w, h = map(int, coords.split(','))

                # Crop the image
                cropped = img[y:y+h, x:x+w]

                # Display the cropped image
                display_image(cropped)

                print("Save this crop? (yes/no)")
                save_decision = input().strip().lower()

                if save_decision == 'yes':
                    extracted_count += 1
                    img_filename = f"{output_dir}/seal_{page_num}_{extracted_count}.png"
                    cv2.imwrite(img_filename, cv2.cvtColor(cropped, cv2.COLOR_RGB2BGR))
                    print(f"Saved as {img_filename}")

                print("Continue cropping on this page? (yes/no)")
                continue_decision = input().strip().lower()
                if continue_decision != 'yes':
                    break
            except:
                print("Invalid format. Use 'x,y,width,height' or 'skip'")

    print(f"Manual extraction complete. {extracted_count} images saved.")
    return output_dir

In [4]:
# CELL 7: Main Execution Function
def main():
    print("=== Mohenjo-daro Seal Image Extractor ===")

    # Step 1: Upload PDF
    pdf_path = upload_pdf()

    # Step 2: Set page range
    print("\nEnter the starting page number (default: 468):")
    start_page_input = input().strip()
    start_page = int(start_page_input) if start_page_input.isdigit() else 468

    print("Enter the ending page number (default: 831):")
    end_page_input = input().strip()
    end_page = int(end_page_input) if end_page_input.isdigit() else 831

    # Step 3: Extract images from PDF
    extracted_images = extract_images_from_pdf(pdf_path, start_page, end_page)

    # Step 4: Choose extraction method
    print("\nChoose extraction method:")
    print("1. Automatic extraction (faster but less accurate)")
    print("2. Semi-automatic extraction with verification")
    print("3. Fully manual extraction")

    method = input().strip()

    if method == "1":
        output_dir = enhanced_seal_extraction(extracted_images)
    elif method == "2":
        output_dir = process_pages(extracted_images)
    else:
        output_dir = manual_cropping_interface(extracted_images)

    # Step 5: Download the extracted images
    print("\nPreparing images for download...")

    # Create a ZIP file of all extracted images
    zip_path = "extracted_mohenjo_daro_seals.zip"
    shutil.make_archive("extracted_mohenjo_daro_seals", 'zip', output_dir)

    # Trigger download
    files.download(zip_path)

    print("Done! Your extracted images have been packaged for download.")


In [11]:
# CELL 8: Run the Program
main()

=== Mohenjo-daro Seal Image Extractor ===
Please upload your PDF file...


Saving Corpus of Indus Seals and Inscriptions. Collections in India.pdf to Corpus of Indus Seals and Inscriptions. Collections in India (1).pdf
PDF 'Corpus of Indus Seals and Inscriptions. Collections in India (1).pdf' uploaded successfully!

Enter the starting page number (default: 468):
468
Enter the ending page number (default: 831):
831
Processing pages 468 to 831 out of 862 total pages...


  0%|          | 0/364 [00:00<?, ?it/s]


Choose extraction method:
1. Automatic extraction (faster but less accurate)
2. Semi-automatic extraction with verification
3. Fully manual extraction
1
Automatic extraction complete. 1631 potential seals extracted.

Preparing images for download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Done! Your extracted images have been packaged for download.
